In [1]:
import os
import pandas as pd
import numpy as np
import json
import glob
import ast

REPO_DIR = os.path.join("/Users/haya1/Documents/LanguageModel_Labels/congressional_bills/")
# REPO_DIR = "."
os.chdir(REPO_DIR)
data_dir = os.path.join(REPO_DIR, "Data/Prediction")
temp_dir = os.path.join(REPO_DIR, "Temp/Prediction")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def euclidean_distance(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.linalg.norm(a - b)

In [2]:
embedded_descriptions_path = os.path.join(temp_dir, "Description/embedded_descriptions.csv")
embedded_descriptions_llm_path = os.path.join(temp_dir, "DescriptionLLM/embedded_descriptions_llm.csv")

bills_llm_completion = pd.read_csv(os.path.join(data_dir, f"bills_llm_completion.csv"))
embedded_descriptions = pd.read_csv(embedded_descriptions_path, converters={"DescriptionEmbed":ast.literal_eval})

KeyboardInterrupt: 

In [4]:

embedded_descriptions_llm = pd.read_csv(embedded_descriptions_llm_path, converters={"DescriptionLLMEmbed":ast.literal_eval})
bills_llm_completion_embed = bills_llm_completion.merge(embedded_descriptions, on="ID").merge(embedded_descriptions_llm, on="ID")

In [75]:
print(len(bills_llm_completion_embed))

39999


In [6]:
bills_llm_completion_embed.reset_index(inplace=True)

In [101]:
import random

def randomly_paired_similarity(df, N=10e3):
    true_sample = df[["DescriptionEmbed"]].sample(n=int(N), replace=True).reset_index(drop=True)
    pred_sample = df[["DescriptionLLMEmbed"]].sample(n=int(N), replace=True).reset_index(drop=True)
    random_pairs = pd.concat([true_sample, pred_sample], axis=1)

    random_pairs["CosineSimilarityRandomPairs"] = random_pairs.apply(lambda x: cosine_similarity(x["DescriptionEmbed"], x["DescriptionLLMEmbed"]), axis=1)
    random_pairs["EuclideanDistanceRandomPairs"] = random_pairs.apply(lambda x: euclidean_distance(x["DescriptionEmbed"], x["DescriptionLLMEmbed"]), axis=1)
    random_pairs = random_pairs[["CosineSimilarityRandomPairs", "EuclideanDistanceRandomPairs"]]
    return(random_pairs.mean())


random.seed(123)
random_pairs = randomly_paired_similarity(bills_llm_completion_embed)

random.seed(123)
random_pairs_group = bills_llm_completion_embed.groupby(['Model', 'AddIntrDate'])[['Model', 'AddIntrDate', 'DescriptionEmbed', 'DescriptionLLMEmbed']].apply(randomly_paired_similarity)

bills_llm_completion_embed["CosineSimilarity"] = bills_llm_completion_embed.apply(lambda x: cosine_similarity(x["DescriptionEmbed"], x["DescriptionLLMEmbed"]), axis=1)
bills_llm_completion_embed["EuclideanDistance"] = bills_llm_completion_embed.apply(lambda x: euclidean_distance(x["DescriptionEmbed"], x["DescriptionLLMEmbed"]), axis=1)

print(random_pairs)
print(bills_llm_completion_embed[['CosineSimilarity', 'EuclideanDistance']].mean())

print(random_pairs_group)
print(bills_llm_completion_embed.groupby(['Model', 'AddIntrDate'])[['Model', 'AddIntrDate', 'CosineSimilarity', 'EuclideanDistance']].mean(numeric_only=True))
# random_pairs_path = os.path.join(data_dir, "random_pairs.csv")
# random_pairs.to_csv(random_pairs_path, index=False)
# print(f"Saved {os.path.basename(random_pairs_path)}, n = {len(random_pairs)}, at {os.path.dirname(random_pairs_path)}")


CosineSimilarityRandomPairs     0.255163
EuclideanDistanceRandomPairs    1.217294
dtype: float64
CosineSimilarity     0.536192
EuclideanDistance    0.931609
dtype: float64
                                CosineSimilarityRandomPairs  \
Model              AddIntrDate                                
gpt-3.5-turbo-0125 False                           0.256148   
                   True                            0.257127   
gpt-4o-2024-05-13  False                           0.260494   
                   True                            0.256343   

                                EuclideanDistanceRandomPairs  
Model              AddIntrDate                                
gpt-3.5-turbo-0125 False                            1.216376  
                   True                             1.215659  
gpt-4o-2024-05-13  False                            1.212714  
                   True                             1.216334  
                                AddIntrDate  CosineSimilarity  \
Model 

## Exact Matches Rate

In [16]:


bills_llm_completion_embed["Identical"] = bills_llm_completion_embed["DescriptionRaw"]==bills_llm_completion_embed["DescriptionLLMRaw"]
identical_grouped = bills_llm_completion_embed.groupby(['Model','AddIntrDate'])["Identical"].mean()*100
identical = bills_llm_completion_embed["Identical"].mean()*100

print(identical_grouped)
print(identical)

Model               AddIntrDate
gpt-3.5-turbo-0125  False          1.480000
                    True           1.550000
gpt-4o-2024-05-13   False          3.920000
                    True           3.470347
Name: Identical, dtype: float64
2.605065126628166


In [15]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [ ]:
bills_llm_completion_embed["CosineSimilarity"] = bills_llm_completion_embed.apply(lambda x: cosine_similarity(x["DescriptionEmbed"], x["DescriptionLLMEmbed"]), axis=1)
bills_llm_completion_embed["EuclideanDistance"] = bills_llm_completion_embed.apply(lambda x: euclidean_distance(x["DescriptionEmbed"], x["DescriptionLLMEmbed"]), axis=1)

In [47]:
bills_llm_completion_similarity = bills_llm_completion_embed[[
    'ID', 'BillID', 'PromptingStrategyID', 'PromptingStrategyName',
    'ResponseFormat', 'TrimText', 'AddIntrDate', 'Model', 'Temperature',
    'MaxTokens', 'InputTokens', 'OutputTokens', 
    'Year', 'Major', 'MajorText', 'Party', 'Chamber', 'DW1', 'PassH', 'PassS', 'Postal', 'IntrDate',
    'DescriptionTrim', 'Description', 'DescriptionLLM', 
    'CosineSimilarity', 'EuclideanDistance', 'Identical']]
bills_llm_completion_similarity_path = os.path.join(data_dir, "bills_llm_completion_similarity.csv")
bills_llm_completion_similarity.to_csv(bills_llm_completion_similarity_path, index=False)
print(f"Saved {os.path.basename(bills_llm_completion_similarity_path)}, n = {len(bills_llm_completion_similarity)}, at {os.path.dirname(bills_llm_completion_similarity_path)}")

Saved bills_llm_completion_similarity.csv, n = 2461, at /Users/haya1/Documents/LanguageModel_Labels/congressional_bills/Data/Prediction
